## Install Dependencies

In [ ]:
# Install dependencies
%pip install -q amplpy matplotlib

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["gurobi", "xpress", "highs", "gokestrel"], # modules to install
    license_uuid="default") # license to use

### AMPL Formulation

In [9]:
%%ampl_eval
reset;

# Sets
set MINREQ;   # nutrients with lower bound requirement
set MAXREQ;   # nutrients with upper bound requirement
set FOOD;     # food items

# Parameters
param amt {MINREQ union MAXREQ, FOOD};  # nutrient composition matrix
param n_max {MAXREQ};                   # upper bound for nutrients

# Decision variable
var Buy {FOOD} >= 0;

# Derived set: nutrients that appear in BOTH MINREQ and MAXREQ
set BOTHREQ = {i in MINREQ: i in MAXREQ};

# Constraint (only created if BOTHREQ is non-empty)
subject to DualLimit {i in BOTHREQ}:
    sum {j in FOOD} amt[i,j] * Buy[j] <= n_max[i];

In [10]:
%%ampl_eval
data;

# Nutrients with lower bounds
set MINREQ := protein fat calcium;

# Nutrients with upper bounds
set MAXREQ := vitamin iron calcium;

# Food set
set FOOD := bread milk apple meat;

# Nutrient composition matrix
param amt:        bread  milk  apple  meat :=
    protein        10     8     2     25
    fat            5      3     1     15
    calcium        20     15    5     10
    vitamin        2      6     10    1
    iron           1      2     3     8;

# Upper bounds ONLY for MAXREQ nutrients
param n_max :=
calcium 60
vitamin 50
iron 40;

display DualLimit;

DualLimit [*] :=
calcium  0
;



### amplpy Formulation
#### AMPL Model

In [12]:
%%ampl_eval
reset;

# Sets
set MINREQ;   # nutrients with lower bound requirement
set MAXREQ;   # nutrients with upper bound requirement
set FOOD;     # food items

# Parameters
param amt {MINREQ union MAXREQ, FOOD};  # nutrient composition matrix
param n_max {MAXREQ};                   # upper bound for nutrients

# Decision variable
var Buy {FOOD} >= 0;

# Derived set: nutrients that appear in BOTH MINREQ and MAXREQ
set BOTHREQ = {i in MINREQ: i in MAXREQ};

# Constraint (only created if BOTHREQ is non-empty)
subject to DualLimit {i in BOTHREQ}:
    sum {j in FOOD} amt[i,j] * Buy[j] <= n_max[i];

#### Python Data

In [15]:
# Sets
ampl.set["MINREQ"].set_values(["protein", "fat", "calcium"])
ampl.set["MAXREQ"].set_values(["vitamin", "iron", "calcium"])
ampl.set["FOOD"].set_values(["bread", "milk", "apple", "meat"])

# Nutrient composition matrix
import pandas as pd

amt = pd.DataFrame({
    "bread": [10, 5, 20, 2, 1],
    "milk":  [8, 3, 15, 6, 2],
    "apple": [2, 1, 5, 10, 3],
    "meat":  [25, 15, 10, 1, 8]
}, index=["protein", "fat", "calcium", "vitamin", "iron"])

# Convert to AMPL-friendly format
amt_dict = {
    (row, col): amt.loc[row, col]
    for row in amt.index
    for col in amt.columns}

# Load matrix once (correct way)
ampl.param["amt"].set_values(amt_dict)

# Upper bounds (only MAXREQ nutrients)
ampl.param["n_max"].set_values({
    "calcium": 60,
    "vitamin": 50,
    "iron": 40})

ampl.eval("display DualLimit;")

DualLimit [*] :=
calcium  0
;

